# GRPO training
Runs `python -m grpo.train` with the arguments in TRAIN_ARGS on GPU 0. Checkpoints, metrics and sampled rollouts go to /kaggle/working/checkpoints.

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "math-verify[antlr4_13_2]" peft 2>&1 | tail -1
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"
!python -m pytest -q tests/ 2>&1 | tail -2

In [ ]:
RUNS = [["--config", "configs/grpo-0.5b.yaml", "--set", "run_name=smoke-0.5b", "steps=10", "eval_every=10", "val_size=100", "save_every=5", "samples_every=10", "out_dir=/kaggle/working/checkpoints"], ["--config", "configs/grpo-0.5b.yaml", "--set", "run_name=smoke-0.5b", "steps=20", "eval_every=10", "val_size=100", "save_every=5", "samples_every=10", "out_dir=/kaggle/working/checkpoints"]]
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "CUDA_VISIBLE_DEVICES": "0"}
for i, args in enumerate(RUNS):
    path = f"/kaggle/working/train-{i}.log"
    code = subprocess.run(["python", "-m", "grpo.train", *args], env=env, stdout=open(path, "w"), stderr=subprocess.STDOUT).returncode
    print(" ".join(args), "-> exit code", code)
    print("".join(l for l in open(path) if l.startswith(("step", "resumed", "trainable", "non-finite")) or "Error" in l)[-6000:])
    if code:
        break
